# v7a  Group Split Experiment

## Purpose
Test if data leakage from the same notification (Nn Notif Nr) appearing in both
train and test sets inflates our scores. We use GroupShuffleSplit to ensure all
samples from one notification stay together.

## Hypothesis
If the same notification appears in both splits, overlapping text gives the model
an unfair advantage  scores may drop with proper group split.

## Dataset
5a_eos_vs_noneos.csv  13,910 samples, 2,741 duplicate notification numbers.

## Kernel: efaai_v3 (Python 3.12)

In [1]:
import os, time, warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier
from scipy.sparse import hstack

warnings.filterwarnings('ignore')
np.random.seed(42)

ROOT = os.path.abspath(os.getcwd())
if not os.path.exists(os.path.join(ROOT, 'data')):
    ROOT = os.path.abspath(os.path.join(ROOT, '..'))

CSV = os.path.join(ROOT, 'data', 'v5a_eos_vs_noneos.csv')
TEXT_COL = 'PSI Failure Desc'
LABEL_COL = 'label'
GROUP_COL = 'Nn Notif Nr'
print(f'ROOT: {ROOT}')

ROOT: <project-root>


In [2]:
# Load and prepare data
df = pd.read_csv(CSV)
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL].str.len() > 3].reset_index(drop=True)
print(f'Loaded: {df.shape}')

le = LabelEncoder()
df['y'] = le.fit_transform(df[LABEL_COL])
EOS_IDX = list(le.classes_).index('EOS')

# Group analysis
groups = df[GROUP_COL].values
n_unique = len(np.unique(groups))
n_dups = len(groups) - n_unique
print(f'Unique notifications: {n_unique}, Duplicates: {n_dups}')
print(f'Samples per group: mean={df.groupby(GROUP_COL).size().mean():.2f}, max={df.groupby(GROUP_COL).size().max()}')

Loaded: (13910, 3)
Unique notifications: 11169, Duplicates: 2741
Samples per group: mean=1.25, max=99


## 2  Standard Split vs Group Split

Check text overlap between train/test in standard random split.

In [3]:
# Standard split (same as v5a/v6)
X_text = df[TEXT_COL].values
y = df['y'].values

X_train_std, X_test_std, y_train_std, y_test_std, idx_train, idx_test = train_test_split(
    X_text, y, np.arange(len(y)), test_size=0.2, stratify=y, random_state=42
)

# Check group leakage in standard split
groups_train = groups[idx_train]
groups_test = groups[idx_test]
overlap = set(groups_train) & set(groups_test)
print(f'Standard split: {len(overlap)} groups appear in BOTH train and test')
print(f'  This means {sum(g in overlap for g in groups_test)} test samples share a notification with training')

# Group split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_text, y, groups=groups))

X_train_grp = X_text[train_idx]
X_test_grp = X_text[test_idx]
y_train_grp = y[train_idx]
y_test_grp = y[test_idx]

overlap_grp = set(groups[train_idx]) & set(groups[test_idx])
print(f'\nGroup split: {len(overlap_grp)} groups in both (should be 0)')
print(f'  Train: {len(train_idx)}, Test: {len(test_idx)}')
print(f'  Train EOS%: {(y_train_grp==EOS_IDX).mean()*100:.1f}%, Test EOS%: {(y_test_grp==EOS_IDX).mean()*100:.1f}%')

Standard split: 518 groups appear in BOTH train and test
  This means 704 test samples share a notification with training

Group split: 0 groups in both (should be 0)
  Train: 11180, Test: 2730
  Train EOS%: 25.7%, Test EOS%: 24.6%


## 3  Model Comparison

Train the v6 best model (word+char TF-IDF + SMOTE + RF) on both split types.

In [4]:
def build_features(X_train, X_test):
    tfidf_word = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=3000, sublinear_tf=True)
    tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=3000, sublinear_tf=True)
    Xw_tr = tfidf_word.fit_transform(X_train)
    Xc_tr = tfidf_char.fit_transform(X_train)
    Xw_te = tfidf_word.transform(X_test)
    Xc_te = tfidf_char.transform(X_test)
    return hstack([Xw_tr, Xc_tr]), hstack([Xw_te, Xc_te])

def evaluate(y_true, y_pred, label=''):
    mf1 = f1_score(y_true, y_pred, average='macro')
    acc = accuracy_score(y_true, y_pred)
    ef1 = f1_score(y_true, y_pred, pos_label=EOS_IDX)
    print(f'  {label}: Macro-F1={mf1:.4f}, Acc={acc:.4f}, EOS-F1={ef1:.4f}')
    return {'label': label, 'macro_f1': mf1, 'accuracy': acc, 'eos_f1': ef1}

results = []

# === Standard Split ===
print("Standard Split (random, may leak groups):")
X_tr_feat, X_te_feat = build_features(X_train_std, X_test_std)
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_tr_feat, y_train_std)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_res, y_res)
preds = rf.predict(X_te_feat)
results.append(evaluate(y_test_std, preds, 'Standard Split + SMOTE + RF'))

# === Group Split ===
print("\nGroup Split (no notification leakage):")
X_tr_feat_g, X_te_feat_g = build_features(X_train_grp, X_test_grp)
sm2 = SMOTE(random_state=42)
X_res_g, y_res_g = sm2.fit_resample(X_tr_feat_g, y_train_grp)
rf_g = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_g.fit(X_res_g, y_res_g)
preds_g = rf_g.predict(X_te_feat_g)
results.append(evaluate(y_test_grp, preds_g, 'Group Split + SMOTE + RF'))

# === Group Split + BalancedRF ===
print("\nGroup Split + BalancedRF (no SMOTE needed):")
brf = BalancedRandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
brf.fit(X_tr_feat_g, y_train_grp)
preds_brf = brf.predict(X_te_feat_g)
results.append(evaluate(y_test_grp, preds_brf, 'Group Split + BalancedRF'))

Standard Split (random, may leak groups):


  Standard Split + SMOTE + RF: Macro-F1=0.7725, Acc=0.8339, EOS-F1=0.6542

Group Split (no notification leakage):


  Group Split + SMOTE + RF: Macro-F1=0.7301, Acc=0.8070, EOS-F1=0.5860

Group Split + BalancedRF (no SMOTE needed):


  Group Split + BalancedRF: Macro-F1=0.7230, Acc=0.7832, EOS-F1=0.5940


In [5]:
# Summary Table
print("=" * 70)
print("  v7a GROUP SPLIT EXPERIMENT  SUMMARY")
print("=" * 70)
rdf = pd.DataFrame(results)
print(rdf.to_string(index=False))

delta = rdf.iloc[1]['macro_f1'] - rdf.iloc[0]['macro_f1']
print(f"\nDelta (Group - Standard): {delta:+.4f} Macro-F1")
if abs(delta) < 0.01:
    print("Conclusion: Minimal difference  group leakage has negligible effect.")
    print("Our standard split results are trustworthy.")
elif delta < -0.01:
    print("Conclusion: Score drops with group split  some inflation from leakage.")
    print("True model performance is slightly lower than reported.")
else:
    print("Conclusion: Group split improves scores  standard split may be harder due to imbalance.")

rdf.to_csv(os.path.join(ROOT, 'results', 'v7a_group_split_results.csv'), index=False)
print("\nSaved: results/v7a_group_split_results.csv")
print("\n v7a complete.")

  v7a GROUP SPLIT EXPERIMENT  SUMMARY


                      label  macro_f1  accuracy   eos_f1
Standard Split + SMOTE + RF  0.772460  0.833932 0.654192
   Group Split + SMOTE + RF  0.730076  0.806960 0.586017
   Group Split + BalancedRF  0.723019  0.783150 0.593964

Delta (Group - Standard): -0.0424 Macro-F1
Conclusion: Score drops with group split  some inflation from leakage.
True model performance is slightly lower than reported.

Saved: results/v7a_group_split_results.csv

 v7a complete.
